# Entrenamiento Donut v3 — Tickets de Mercadona

Cambios respecto a la versión anterior (v2):

| | v2 | v3 |
|---|---|---|
| Ground Truth | parser antiguo: 43 de 91 tickets tenían algún error | se extrae del PDF y **se valida** (suma de importes = TOTAL) |
| Formato de salida | JSON como texto plano (`{"menu": ...}`), 13 tickets truncados a 512 tokens | tokens especiales de Donut (`<s_nm>…</s_nm>`), secuencia máx. 628 tokens |
| Resolución | 960×720: un ticket largo quedaba con letra de ~10 px | 1920×720 (T4) o 2560×960 (L4/A100): letra de 20–30 px |
| Datos | 91 tickets (con 5 duplicados) | 77 train + 14 validación + tickets sintéticos |
| Evaluación | ninguna | métricas por ticket y por producto en validación, mejor checkpoint |

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno → GPU` y sube `datos_donut_v3.zip` a `/content`.

In [ ]:
!pip install -q -U transformers datasets accelerate sentencepiece pdfplumber
# torchaudio no se usa y la versión de Colab está compilada para otra CUDA distinta a la de PyTorch:
# si está instalada, transformers intenta importarla y falla al cargar DonutProcessor.
!pip uninstall -y -q torchaudio
!apt-get -qq install -y fonts-urw-base35 > /dev/null
!unzip -q -o /content/datos_donut_v3.zip -d /content/
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Preparar el dataset real
Genera `dataset_v3/train` y `dataset_v3/validation` a partir de los PDF (determinista, semilla 42).
El informe indica duplicados eliminados y tickets descartados.

In [ ]:
!python /content/datos_donut_v3/preparar_dataset.py /content/datos_donut_v3/pdfs /content/datos_donut_v3/dataset_v3

## 2. Configuración
La resolución se elige automáticamente según la memoria de la GPU. Si te quedas sin memoria (CUDA OOM),
baja `IMG_SIZE` a `(1920, 720)` o `(1600, 600)`.

In [ ]:
import torch, json, random, re, os
from pathlib import Path

BASE = Path("/content/datos_donut_v3")
DATASET = BASE / "dataset_v3"
MODELO_BASE = "naver-clova-ix/donut-base-finetuned-cord-v2"
TASK_TOKEN = "<s_mercadona>"
MAX_LENGTH = 768            # límite de posiciones del decoder; la secuencia más larga del dataset ocupa 628

gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
IMG_SIZE = (2560, 960) if gpu_gb >= 22 else (1920, 720)     # (alto, ancho)
USE_BF16 = torch.cuda.is_bf16_supported()

N_SINTETICOS = 800          # 0 para entrenar solo con tickets reales
REPETIR_REALES = 3          # cada época ve 3 veces cada ticket real (son la distribución objetivo)
EPOCHS = 8
LR = 3e-5
GRAD_ACCUM = 4
SEED = 42

random.seed(SEED); torch.manual_seed(SEED)
print(f"GPU {torch.cuda.get_device_name(0)} ({gpu_gb:.0f} GB) -> IMG_SIZE={IMG_SIZE}, bf16={USE_BF16}")

## 3. Tickets sintéticos (opcional pero recomendado)
Se dibujan tickets con la misma maqueta que los reales, con productos del split de **entrenamiento**
y del catálogo de Mercadona. Su Ground Truth es exacto porque lo generamos nosotros.

In [ ]:
if N_SINTETICOS > 0:
    !python /content/datos_donut_v3/generar_sinteticos.py {DATASET} {BASE}/catalogo_mercadona.csv {N_SINTETICOS} 1234

from PIL import Image
from IPython.display import display
ejemplo = Image.open(DATASET / "synthetic" / "sint_00000.jpg") if N_SINTETICOS > 0 else None
if ejemplo: display(ejemplo.resize((ejemplo.width // 4, ejemplo.height // 4)))

## 4. Carga de datos y conversión a tokens de Donut

In [ ]:
def cargar_split(nombre):
    carpeta = DATASET / nombre
    if not (carpeta / "metadata.jsonl").exists():
        return []
    datos = []
    for linea in open(carpeta / "metadata.jsonl", encoding="utf-8"):
        r = json.loads(linea)
        datos.append({"imagen": str(carpeta / r["file_name"]),
                      "gt": json.loads(r["ground_truth"])["gt_parse"]})
    return datos

train_real = cargar_split("train")
validacion = cargar_split("validation")
sinteticos = cargar_split("synthetic")
train = train_real * REPETIR_REALES + sinteticos
random.shuffle(train)
print(f"Reales train: {len(train_real)} | validación: {len(validacion)} | sintéticos: {len(sinteticos)} | muestras/época: {len(train)}")

MAPEO = {}   # caracteres que el tokenizador no conoce -> equivalente que sí conoce (se rellena en el paso 5)

def json2token(obj):
    if isinstance(obj, dict):
        return "".join(f"<s_{k}>" + json2token(v) + f"</s_{k}>" for k, v in obj.items())
    if isinstance(obj, list):
        return "<sep/>".join(json2token(x) for x in obj)
    return str(obj).translate(str.maketrans(MAPEO))

def claves(obj, acc):
    if isinstance(obj, dict):
        for k, v in obj.items():
            acc.add(k); claves(v, acc)
    elif isinstance(obj, list):
        for x in obj: claves(x, acc)
    return acc

todas_claves = set()
for d in train_real + validacion + sinteticos:
    claves(d["gt"], todas_claves)
tokens_nuevos = [TASK_TOKEN] + sorted({f"<s_{k}>" for k in todas_claves} | {f"</s_{k}>" for k in todas_claves})
print(json2token(train_real[0]["gt"])[:300])

## 5. Modelo y procesador

In [ ]:
from transformers import DonutProcessor, VisionEncoderDecoderModel

processor = DonutProcessor.from_pretrained(MODELO_BASE)
model = VisionEncoderDecoderModel.from_pretrained(MODELO_BASE)

# Tokens especiales: los que ya existen en CORD (<s_nm>, <s_price>...) se reutilizan,
# el resto (<s_date>, <s_weight>, <s_mercadona>) se añaden.
tok = processor.tokenizer
vocab = tok.get_vocab()
faltan = [t for t in tokens_nuevos if t not in vocab]
tok.add_tokens(faltan, special_tokens=True)
model.decoder.resize_token_embeddings(len(tok))
print("Tokens añadidos:", faltan)

# El tokenizador de Donut no tiene "Ñ" ni "È" (los convierte en <unk> y el modelo nunca podría escribirlos).
# Sí conoce "ñ" y "è", así que en el Ground Truth se usan las minúsculas y al decodificar se deshace el cambio
# (los nombres de producto van siempre en mayúsculas, así que no hay ambigüedad).
caracteres = set()
for d in train_real + validacion + sinteticos:
    caracteres |= set(json.dumps(d["gt"], ensure_ascii=False))
for c in sorted(caracteres):
    if tok.unk_token_id in tok(c, add_special_tokens=False).input_ids:
        alt = c.lower()
        if tok.unk_token_id not in tok(alt, add_special_tokens=False).input_ids:
            MAPEO[c] = alt
        else:
            print("⚠️ Carácter sin representación:", repr(c))
MAPEO_INVERSO = {v: k for k, v in MAPEO.items()}
print("Mapeo de caracteres:", MAPEO)

# Resolución
processor.image_processor.size = {"height": IMG_SIZE[0], "width": IMG_SIZE[1]}
processor.image_processor.do_align_long_axis = False
model.config.encoder.image_size = list(IMG_SIZE)
model.config.decoder.max_length = MAX_LENGTH

# Tokens de control
model.config.pad_token_id = tok.pad_token_id
model.config.decoder_start_token_id = tok.convert_tokens_to_ids(TASK_TOKEN)
model.config.eos_token_id = tok.eos_token_id
gc = model.generation_config
gc.decoder_start_token_id = model.config.decoder_start_token_id
gc.pad_token_id = tok.pad_token_id
gc.eos_token_id = tok.eos_token_id
gc.max_length = MAX_LENGTH
gc.num_beams = 1
gc.bad_words_ids = [[tok.unk_token_id]]
gc.use_cache = True

model.gradient_checkpointing_enable()
model.config.use_cache = False

## 6. Dataset con aumentación
Solo en entrenamiento: ligera rotación, desplazamiento, cambio de escala, desenfoque, contraste y compresión JPEG.
Así el modelo no memoriza píxeles exactos y aguanta mejor imágenes algo distintas (capturas, fotos escaneadas…).

In [ ]:
import io
from PIL import ImageFilter, ImageEnhance, ImageOps
from torch.utils.data import Dataset

def aumentar(img):
    img = img.convert("RGB")
    if random.random() < 0.5:
        img = img.rotate(random.uniform(-1.5, 1.5), resample=Image.BICUBIC, expand=True, fillcolor="white")
    if random.random() < 0.5:   # márgenes aleatorios (desplaza el ticket dentro del lienzo)
        w, h = img.size
        img = ImageOps.expand(img, border=(random.randint(0, w // 12), random.randint(0, h // 30),
                                           random.randint(0, w // 12), random.randint(0, h // 30)), fill="white")
    if random.random() < 0.3:
        f = random.uniform(0.6, 1.0)
        img = img.resize((int(img.width * f), int(img.height * f)), Image.BILINEAR)
    if random.random() < 0.2:
        img = img.filter(ImageFilter.GaussianBlur(random.uniform(0.3, 1.0)))
    if random.random() < 0.3:
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.7, 1.2))
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.9, 1.1))
    if random.random() < 0.3:
        buf = io.BytesIO(); img.save(buf, "JPEG", quality=random.randint(40, 85)); img = Image.open(buf).convert("RGB")
    return img

class TicketsDataset(Dataset):
    def __init__(self, datos, entrenamiento):
        self.datos, self.entrenamiento = datos, entrenamiento
    def __len__(self):
        return len(self.datos)
    def __getitem__(self, i):
        d = self.datos[i]
        img = Image.open(d["imagen"]).convert("RGB")
        if self.entrenamiento:
            img = aumentar(img)
        pixel_values = processor(images=img, return_tensors="pt").pixel_values[0]
        objetivo = json2token(d["gt"]) + tok.eos_token
        ids = tok(objetivo, add_special_tokens=False, max_length=MAX_LENGTH, truncation=True).input_ids
        return {"pixel_values": pixel_values, "labels": torch.tensor(ids)}

def collate(batch):
    pv = torch.stack([b["pixel_values"] for b in batch])
    largo = max(len(b["labels"]) for b in batch)
    labels = torch.full((len(batch), largo), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        labels[i, :len(b["labels"])] = b["labels"]
    return {"pixel_values": pv, "labels": labels}

ds_train = TicketsDataset(train, True)
ds_val = TicketsDataset(validacion, False)

largos = [len(tok(json2token(d["gt"]) + tok.eos_token, add_special_tokens=False).input_ids) for d in train_real + validacion + sinteticos]
print("Longitud máxima de secuencia:", max(largos), "(límite", MAX_LENGTH, ")")
assert max(largos) <= MAX_LENGTH, "Hay secuencias que se truncarían"

## 7. Métricas
- **ticket_perfecto**: fecha, total y todos los productos (nombre, cantidad, precios, peso) exactos.
- **ticket_casi_perfecto**: fecha y total correctos y como mucho 2 productos con fallo (criterio del objetivo del TFG).
- **f1_productos**: F1 sobre las tuplas (nombre, cantidad, importe).
- **acierto_total / acierto_fecha**.

In [ ]:
from collections import Counter

def token2json(tokens, interno=False):
    # Adaptado del repositorio oficial de Donut (clovaai/donut, licencia MIT)
    salida = {}
    while tokens:
        inicio = re.search(r"<s_(.*?)>", tokens, re.IGNORECASE)
        if inicio is None:
            break
        clave = inicio.group(1)
        fin = re.search(rf"</s_{re.escape(clave)}>", tokens, re.IGNORECASE)
        inicio = inicio.group()
        if fin is None:
            tokens = tokens.replace(inicio, "")
            continue
        fin = fin.group()
        contenido = re.search(f"{re.escape(inicio)}(.*?){re.escape(fin)}", tokens, re.IGNORECASE | re.DOTALL)
        if contenido is not None:
            contenido = contenido.group(1).strip()
            if "<s_" in contenido and "</s_" in contenido:
                valor = token2json(contenido, interno=True)
                if valor:
                    salida[clave] = valor[0] if len(valor) == 1 else valor
            else:
                hojas = [h.strip() for h in contenido.split("<sep/>")]
                salida[clave] = hojas[0] if len(hojas) == 1 else hojas
        tokens = tokens[tokens.find(fin) + len(fin):].strip()
        if tokens[:6] == "<sep/>":
            return [salida] + token2json(tokens[6:], interno=True)
    if salida:
        return [salida] if interno else salida
    return [] if interno else {"text_sequence": tokens}

def decodificar(ids):
    texto = tok.decode(ids, skip_special_tokens=False)
    texto = texto.replace(tok.eos_token, "").replace(tok.pad_token, "").replace(TASK_TOKEN, "").replace("<s>", "")
    texto = texto.translate(str.maketrans(MAPEO_INVERSO))
    return token2json(texto.strip())

def lista_menu(gt):
    m = gt.get("menu", []) if isinstance(gt, dict) else []
    return [m] if isinstance(m, dict) else [x for x in m if isinstance(x, dict)]

def total_de(gt):
    t = gt.get("total", {}) if isinstance(gt, dict) else {}
    return t.get("total_price") if isinstance(t, dict) else None

def comparar(pred, gt):
    p_items = Counter(json.dumps(x, sort_keys=True, ensure_ascii=False) for x in lista_menu(pred))
    g_items = Counter(json.dumps(x, sort_keys=True, ensure_ascii=False) for x in lista_menu(gt))
    p_t = Counter((x.get("nm"), x.get("cnt"), x.get("price")) for x in lista_menu(pred))
    g_t = Counter((x.get("nm"), x.get("cnt"), x.get("price")) for x in lista_menu(gt))
    tp = sum((p_t & g_t).values())
    errores = max(sum((p_items - g_items).values()), sum((g_items - p_items).values()))
    ok_total = total_de(pred) == total_de(gt)
    ok_fecha = pred.get("date") == gt.get("date")
    return {"tp": tp, "n_pred": sum(p_t.values()), "n_gt": sum(g_t.values()), "errores_items": errores,
            "ok_total": ok_total, "ok_fecha": ok_fecha,
            "perfecto": ok_total and ok_fecha and errores == 0 and lista_menu(pred) == lista_menu(gt),
            "casi_perfecto": ok_total and ok_fecha and errores <= 2}

def resumen(comparaciones):
    n = len(comparaciones)
    tp = sum(c["tp"] for c in comparaciones)
    prec = tp / max(1, sum(c["n_pred"] for c in comparaciones))
    rec = tp / max(1, sum(c["n_gt"] for c in comparaciones))
    return {"ticket_perfecto": sum(c["perfecto"] for c in comparaciones) / n,
            "ticket_casi_perfecto": sum(c["casi_perfecto"] for c in comparaciones) / n,
            "f1_productos": 2 * prec * rec / max(1e-9, prec + rec),
            "precision_productos": prec, "recall_productos": rec,
            "acierto_total": sum(c["ok_total"] for c in comparaciones) / n,
            "acierto_fecha": sum(c["ok_fecha"] for c in comparaciones) / n}

def compute_metrics(eval_pred):
    preds = eval_pred.predictions
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = [[t for t in fila if t >= 0] for fila in preds]
    comps = [comparar(decodificar(p), d["gt"]) for p, d in zip(preds, validacion)]
    return resumen(comps)

## 8. Entrenamiento
Se evalúa en validación al final de cada época y se guarda el mejor modelo según `f1_productos`.
Si la métrica no mejora en 3 épocas seguidas, se para (early stopping).

> Consejo: monta Google Drive y cambia `output_dir` a una carpeta de Drive si temes que Colab se desconecte.

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback

args = Seq2SeqTrainingArguments(
    output_dir="/content/checkpoints_v3",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=max(1, int(0.05 * EPOCHS * len(train) / GRAD_ACCUM)),
    weight_decay=0.01,
    bf16=USE_BF16,
    fp16=not USE_BF16,
    gradient_checkpointing=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_productos",
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
    generation_num_beams=1,
    logging_steps=20,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    report_to="none",
    seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=collate,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
trainer.train()

## 9. Evaluación final en validación (para la memoria)

In [ ]:
import pandas as pd

model.eval(); model.config.use_cache = True
device = "cuda"
filas, comps = [], []
for d in validacion:
    img = Image.open(d["imagen"]).convert("RGB")
    pv = processor(images=img, return_tensors="pt").pixel_values.to(device, dtype=model.dtype)
    with torch.no_grad():
        out = model.generate(pv, generation_config=model.generation_config)
    pred = decodificar(out[0].tolist())
    c = comparar(pred, d["gt"]); comps.append(c)
    filas.append({"imagen": Path(d["imagen"]).name, "productos_gt": c["n_gt"], "productos_pred": c["n_pred"],
                  "errores_productos": c["errores_items"], "total_ok": c["ok_total"], "fecha_ok": c["ok_fecha"],
                  "perfecto": c["perfecto"], "casi_perfecto": c["casi_perfecto"],
                  "prediccion": json.dumps(pred, ensure_ascii=False)})

metricas = resumen(comps)
tabla = pd.DataFrame(filas)
tabla.to_csv("/content/resultados_validacion_v3.csv", index=False)
json.dump({"metricas": metricas, "img_size": IMG_SIZE, "n_sinteticos": N_SINTETICOS, "epochs": EPOCHS,
           "historial": trainer.state.log_history}, open("/content/metricas_v3.json", "w"), ensure_ascii=False, indent=2)
print(json.dumps(metricas, indent=2))
display(tabla.drop(columns=["prediccion"]))

## 10. Guardar y descargar el modelo

In [ ]:
SALIDA = "/content/mi_modelo_mercadona_v3"
model.config.use_cache = True
model.save_pretrained(SALIDA)
processor.save_pretrained(SALIDA)
json.dump({"task_token": TASK_TOKEN, "mapeo_inverso": MAPEO_INVERSO},
          open(f"{SALIDA}/donut_mercadona.json", "w"), ensure_ascii=False)
!cd /content && zip -q -r mi_modelo_mercadona_v3.zip mi_modelo_mercadona_v3 metricas_v3.json resultados_validacion_v3.csv
print("📦 /content/mi_modelo_mercadona_v3.zip listo")

# Opcional: copiar a Google Drive (más fiable que la descarga directa para ~1 GB)
# from google.colab import drive; drive.mount('/content/drive')
# !cp /content/mi_modelo_mercadona_v3.zip /content/drive/MyDrive/

## 11. (Opcional) Comparar con el modelo anterior
Sube `mi_modelo_mercadona_v2.zip` a `/content` para medir el modelo antiguo **sobre los mismos 14 tickets de validación**
con las mismas métricas. Así tienes la comparación antes/después para el capítulo de Resultados.

⚠️ El modelo antiguo se entrenó con tickets que pueden estar en esta validación, así que su resultado sale *optimista*.

In [ ]:
ZIP_V2 = "/content/mi_modelo_mercadona_v2.zip"
if os.path.exists(ZIP_V2):
    !unzip -q -o {ZIP_V2} -d /content/
    p2 = DonutProcessor.from_pretrained("/content/mi_modelo_mercadona_v2")
    m2 = VisionEncoderDecoderModel.from_pretrained("/content/mi_modelo_mercadona_v2").to(device).eval()
    def a_formato_v3(texto):
        # El modelo v2 devolvía JSON en texto plano: {"gt_parse": {"store_info":..., "menu":[...], "total":...}}
        try:
            d = json.loads(texto[texto.find("{"):])
        except Exception:
            return {}
        d = d.get("gt_parse", d)
        return {"date": (d.get("store_info") or {}).get("date"), "menu": d.get("menu", []), "total": d.get("total", {})}
    comps2 = []
    for d in validacion:
        img = Image.open(d["imagen"]).convert("RGB")
        pv = p2(images=img, return_tensors="pt").pixel_values.to(device)
        ids = p2.tokenizer("<s_cord-v2>", add_special_tokens=False, return_tensors="pt").input_ids.to(device)
        with torch.no_grad():
            out = m2.generate(pv, decoder_input_ids=ids, max_length=768, num_beams=1,
                              pad_token_id=p2.tokenizer.pad_token_id, eos_token_id=p2.tokenizer.eos_token_id)
        texto = p2.batch_decode(out)[0].replace(p2.tokenizer.eos_token, "").replace(p2.tokenizer.pad_token, "")
        texto = texto.replace("<s_cord-v2>", "")
        gt_v2 = {"date": d["gt"]["date"], "total": d["gt"]["total"],
                 # En v2 "price" era el precio unitario cuando la cantidad era > 1
                 "menu": [{"nm": x["nm"], "cnt": x["cnt"],
                           "price": x["unitprice"] if ("unitprice" in x and "weight" not in x) else x["price"]}
                          for x in d["gt"]["menu"]]}
        pred = a_formato_v3(texto)
        pred["menu"] = [{"nm": x.get("nm"), "cnt": x.get("cnt"), "price": x.get("price")} for x in lista_menu(pred)]
        comps2.append(comparar(pred, gt_v2))
    print("Modelo v2:", json.dumps(resumen(comps2), indent=2))
    print("Modelo v3:", json.dumps(metricas, indent=2))
else:
    print("No se encontró", ZIP_V2, "- se omite la comparación")